### self attention

In [172]:
import torch
inputs=torch.tensor([[0.22,0.44,0.56], # your 
                    [0.33,-0.144,0.756], # journey
                    [0.22,0.344,0.856], # starts
                    [-0.722,-0.644,-0.356], # with
                    [0.022,0.744,0.1356], # one
                    [0.122,0.044,0.956]]) # step 

## key,value,query matrix 
these are the trainable matrix 

In [173]:
w_k=torch.tensor([[0.42,0.232],
                [0.22,0.4232],
                [0.142,0.9232]]
                 )

In [174]:
w_v=torch.tensor([[0.142,0.2232],
                [0.122,0.64232],
                [0.0142,0.29232]]
                 )

In [175]:
w_q=torch.tensor([[0.142,0.9232],
                [0.222,0.84232],
                [0.1142,0.59232]]
                 )

## 1. convert embedding vector -> input k,q,v

In [176]:
input_wk=inputs@w_k ## (6,3)(3,2)-> (6,2) matrix
input_wv=inputs@w_v ## (6,3)(3,2)-> (6,2) matrix
input_wq=inputs@w_q ## (6,3)(3,2)-> (6,2) matrix

In [177]:
print(input_wk)

tensor([[ 0.2687,  0.7542],
        [ 0.2143,  0.7136],
        [ 0.2896,  0.9869],
        [-0.4955, -0.7687],
        [ 0.1922,  0.4452],
        [ 0.1967,  0.9295]])


now only focus on the k,v,q matrix we do all th things on this 

### 2. Compute Attention Score 

In [178]:
query_2=input_wq[1] ## query of journey= 2nd word in the sentence
print(query_2.shape) ## 1-D tensor it is 
print(input_wk.shape)
attention_score_2=input_wk@query_2 ## (6,2)(2,)->(6,) matrix
print(attention_score_2) # 1-D tensor 
print(attention_score_2.shape)
attention_score_2=query_2@input_wk.T 
print(attention_score_2)
print(attention_score_2.shape)

torch.Size([2])
torch.Size([6, 2])
tensor([ 0.5032,  0.4721,  0.6522, -0.5353,  0.3004,  0.6066])
torch.Size([6])
tensor([ 0.5032,  0.4721,  0.6522, -0.5353,  0.3004,  0.6066])
torch.Size([6])


In [179]:
attention_score=input_wq@input_wk.T
print(attention_score)
print(attention_score[0])
attention_score_x=input_wk@input_wq.T
print(attention_score_x)
print(attention_score_x[:][0])

tensor([[ 0.7347,  0.6874,  0.9494, -0.7916,  0.4401,  0.8795],
        [ 0.5032,  0.4721,  0.6522, -0.5353,  0.3004,  0.6066],
        [ 0.8093,  0.7575,  1.0462, -0.8704,  0.4846,  0.9698],
        [-1.1478, -1.0745, -1.4841,  1.2332, -0.6870, -1.3761],
        [ 0.5980,  0.5584,  0.7710, -0.6501,  0.3591,  0.7122],
        [ 0.5766,  0.5401,  0.7460, -0.6179,  0.3449,  0.6923]])
tensor([ 0.7347,  0.6874,  0.9494, -0.7916,  0.4401,  0.8795])
tensor([[ 0.7347,  0.5032,  0.8093, -1.1478,  0.5980,  0.5766],
        [ 0.6874,  0.4721,  0.7575, -1.0745,  0.5584,  0.5401],
        [ 0.9494,  0.6522,  1.0462, -1.4841,  0.7710,  0.7460],
        [-0.7916, -0.5353, -0.8704,  1.2332, -0.6501, -0.6179],
        [ 0.4401,  0.3004,  0.4846, -0.6870,  0.3591,  0.3449],
        [ 0.8795,  0.6066,  0.9698, -1.3761,  0.7122,  0.6923]])
tensor([ 0.7347,  0.5032,  0.8093, -1.1478,  0.5980,  0.5766])


### Attention Weights 

In [180]:
## 1. scale by the sqrt(d_key)
d_key=input_wk.shape[-1]
print(d_key)

2


In [181]:
attention_weights=torch.softmax(attention_score/d_key**0.5,dim=-1)
print(attention_weights)


tensor([[0.1855, 0.1794, 0.2159, 0.0630, 0.1506, 0.2055],
        [0.1814, 0.1775, 0.2016, 0.0871, 0.1572, 0.1952],
        [0.1864, 0.1796, 0.2203, 0.0568, 0.1481, 0.2087],
        [0.0956, 0.1007, 0.0753, 0.5147, 0.1324, 0.0813],
        [0.1837, 0.1786, 0.2076, 0.0760, 0.1551, 0.1991],
        [0.1830, 0.1783, 0.2062, 0.0786, 0.1553, 0.1986]])


### 4. context vector

In [182]:
attention_weights_2=attention_weights[1] #(6,)
 # input_wv (6,2)
context_vector_2=attention_weights_2@input_wv
print(context_vector_2)

tensor([0.0471, 0.3191])


In [183]:
context_vector=attention_weights@input_wv
print(context_vector)

tensor([[ 0.0530,  0.3452],
        [ 0.0471,  0.3191],
        [ 0.0545,  0.3520],
        [-0.0608, -0.1461],
        [ 0.0499,  0.3312],
        [ 0.0492,  0.3283]])


### 6. Implement Python class

In [184]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        ## 1. convert embedding -> k,v,q
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        ## 2. attention score
        attn_scores = queries @ keys.T # omega
        ## 3. attention weights
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        ### 4. context vector
        context_vec = attn_weights @ values
        return context_vec

In [185]:
torch.manual_seed(123)
d_in=3
d_out=2
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[ 0.1531,  0.6076],
        [ 0.1449,  0.5838],
        [ 0.1593,  0.6251],
        [-0.0289,  0.0869],
        [ 0.1426,  0.5782],
        [ 0.1525,  0.6052]], grad_fn=<MmBackward0>)
